# Distributed Training Demonstration

This notebook provides a comprehensive walkthrough of the Horovod distributed training pipeline for transformer language models.

## Overview

This notebook demonstrates:
1. **Configuration Management** - Loading and understanding training configurations
2. **Model Architecture** - Custom TransformerLM and HuggingFace model comparison
3. **Data Pipeline** - Dataset loading, tokenization, and distributed sampling
4. **Training Loop** - Step-by-step explanation of the training process
5. **Horovod Integration** - Distributed training setup and synchronization
6. **Results Analysis** - Training curves, metrics, and model evaluation
7. **Text Generation** - Loading checkpoints and generating text

**Note:** This notebook demonstrates the training process with simplified examples. Full distributed training with multiple GPUs is executed via `scripts/train_zaratan.sh` on the HPC cluster.

## Project Structure

```
src/
├── train.py              # Main training script (Horovod distributed)
├── data.py               # Data loading with DistributedSampler
├── models/
│   ├── transformer_lm.py # Custom GPT-like transformer
│   └── hf_wrapper.py      # HuggingFace model wrapper
└── utils/
    ├── distributed.py     # Horovod helpers
    ├── config.py          # Configuration management
    └── logging.py         # Logging utilities
```


## 1. Setup and Imports


In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import GPT2TokenizerFast
import yaml

# Import project modules
from src.utils.config import load_config
from src.models.transformer_lm import TransformerLM
from src.models.hf_wrapper import load_hf_model, get_model_from_config
from src.data import get_dataloaders, load_and_tokenize_dataset
from src.metrics import compute_perplexity, MetricsTracker

print("[OK] All imports successful!")
print(f"[OK] Project root: {project_root}")
print(f"[OK] PyTorch version: {torch.__version__}")
print(f"[OK] CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[OK] GPU: {torch.cuda.get_device_name(0)}")


## 2. Configuration Management

The training pipeline uses YAML configuration files for easy experimentation. Let's examine the base configuration:


In [ ]:
# Load configuration
config_path = project_root / "configs" / "base.yaml"
config = load_config(str(config_path))

print("=" * 60)
print("Training Configuration")
print("=" * 60)
print(f"\nModel Configuration:")
print(f"  Type: {config.model.type}")
print(f"  d_model: {config.model.d_model}")
print(f"  n_heads: {config.model.n_heads}")
print(f"  n_layers: {config.model.n_layers}")
print(f"  d_ff: {config.model.d_ff}")
print(f"  vocab_size: {config.model.vocab_size}")
print(f"  max_seq_len: {config.model.max_seq_len}")

print(f"\nTraining Configuration:")
print(f"  Epochs: {config.training.epochs}")
print(f"  Per-GPU batch size: {config.training.per_gpu_batch_size}")
print(f"  Learning rate: {config.training.learning_rate}")
print(f"  Warmup steps: {config.training.warmup_steps}")
print(f"  Weight decay: {config.training.weight_decay}")
print(f"  Max grad norm: {config.training.max_grad_norm}")

print(f"\nData Configuration:")
print(f"  Dataset: {config.data.dataset_name}")
print(f"  Validation split: {config.data.validation_split}")
print(f"  Num workers: {config.data.get('num_workers', 4)}")

print(f"\nPaths:")
print(f"  Checkpoints: {config.paths.checkpoint_dir}")
print(f"  Logs: {config.paths.log_dir}")
print(f"  TensorBoard: {config.paths.tensorboard_dir}")


## 3. Model Architecture

Let's examine the custom TransformerLM architecture and compare it with HuggingFace models.


In [ ]:
# Create custom model
custom_model = TransformerLM(
    vocab_size=config.model.vocab_size,
    d_model=config.model.d_model,
    n_heads=config.model.n_heads,
    n_layers=config.model.n_layers,
    d_ff=config.model.d_ff,
    max_seq_len=config.model.max_seq_len,
    dropout=config.model.dropout
)

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

custom_params = count_parameters(custom_model)
print("=" * 60)
print("Custom TransformerLM")
print("=" * 60)
print(f"Total parameters: {custom_params:,}")
print(f"Model size: {custom_params / 1e6:.2f}M parameters")
print(f"\nArchitecture:")
print(f"  Embedding dimension: {custom_model.d_model}")
print(f"  Attention heads: {custom_model.n_heads}")
print(f"  Transformer layers: {custom_model.n_layers}")
print(f"  Feed-forward dimension: {config.model.d_ff}")
print(f"  Vocabulary size: {custom_model.vocab_size}")


In [ ]:
# Compare with HuggingFace models
print("\n" + "=" * 60)
print("Model Comparison")
print("=" * 60)

models_to_compare = {
    "Custom TransformerLM": custom_model,
}

# Add HuggingFace models for comparison
try:
    distilgpt2 = load_hf_model("distilgpt2")
    gpt2_small = load_hf_model("gpt2")
    
    models_to_compare["DistilGPT-2"] = distilgpt2
    models_to_compare["GPT-2 Small"] = gpt2_small
    
    print(f"\n{'Model':<25} {'Parameters':<15} {'Size (M)':<10}")
    print("-" * 50)
    for name, model in models_to_compare.items():
        params = count_parameters(model)
        size_m = params / 1e6
        print(f"{name:<25} {params:>14,} {size_m:>9.2f}M")
except Exception as e:
    print(f"Note: Could not load HuggingFace models: {e}")
    print("This is expected if models are not cached.")


### Model Architecture Visualization

The custom TransformerLM follows a GPT-like decoder-only architecture:

```
Input Tokens
    ↓
Token Embedding (vocab_size × d_model)
    ↓
Positional Encoding (sinusoidal)
    ↓
┌─────────────────────┐
│ Transformer Block 1 │
│  - Multi-Head Attn  │
│  - Feed Forward     │
│  - Layer Norm       │
└─────────────────────┘
    ↓
┌─────────────────────┐
│ Transformer Block 2 │
│  ...                │
└─────────────────────┘
    ↓
    ...
    ↓
┌─────────────────────┐
│ Transformer Block N │
└─────────────────────┘
    ↓
Final Layer Norm
    ↓
Output Projection (d_model × vocab_size)
    ↓
Logits (vocab_size)
```

**Key Features:**
- **Causal Masking**: Ensures autoregressive property (can't see future tokens)
- **Weight Tying**: Shared weights between embedding and output projection
- **Pre-norm Architecture**: Layer normalization before attention/FFN
- **GELU Activation**: Gaussian Error Linear Unit in feed-forward networks


## 4. Data Pipeline

Let's examine the data loading and preprocessing pipeline:


In [ ]:
# Load tokenizer
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

print("=" * 60)
print("Tokenizer Information")
print("=" * 60)
print(f"Vocabulary size: {len(tokenizer)}")
print(f"Special tokens:")
print(f"  BOS: {tokenizer.bos_token} (ID: {tokenizer.bos_token_id})")
print(f"  EOS: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"  PAD: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

# Example tokenization
sample_text = "The quick brown fox jumps over the lazy dog."
tokens = tokenizer(sample_text, return_tensors='pt')
print(f"\nExample tokenization:")
print(f"  Text: '{sample_text}'")
print(f"  Token IDs: {tokens['input_ids'][0].tolist()}")
print(f"  Decoded: {tokenizer.decode(tokens['input_ids'][0])}")


In [ ]:
# Demonstrate data loading (with small subset for notebook)
print("\n" + "=" * 60)
print("Data Loading Demonstration")
print("=" * 60)

# Create a modified config for quick demo
demo_config = config
demo_config.data.max_samples = 1000  # Small subset for demo

try:
    # Load dataset (this will use synthetic data if real dataset unavailable)
    train_dataset, val_dataset = load_and_tokenize_dataset(
        dataset_name=config.data.dataset_name,
        max_samples=1000,  # Small subset for demo
        validation_split=config.data.validation_split,
        max_length=config.model.max_seq_len,
        cache_dir=os.environ.get('HF_HOME', None)
    )
    
    print(f"[OK] Train dataset: {len(train_dataset)} samples")
    print(f"[OK] Validation dataset: {len(val_dataset)} samples")
    
    # Show example batch
    sample = train_dataset[0]
    print(f"\nExample batch structure:")
    print(f"  input_ids shape: {sample['input_ids'].shape}")
    print(f"  attention_mask shape: {sample['attention_mask'].shape}")
    print(f"  labels shape: {sample['labels'].shape}")
    print(f"  First 20 tokens: {sample['input_ids'][:20].tolist()}")
    
except Exception as e:
    print(f"Note: Dataset loading failed (expected in notebook): {e}")
    print("In actual training, this uses the full BookCorpus dataset.")


## 5. Training Loop Explanation

The training process follows these key steps:

### 5.1 Training Step Breakdown

For each batch:
1. **Forward Pass**: Compute logits and loss
2. **Backward Pass**: Compute gradients
3. **Gradient Clipping**: Prevent exploding gradients
4. **Optimizer Step**: Update parameters
5. **Scheduler Step**: Update learning rate
6. **Metric Averaging**: Synchronize metrics across GPUs (Horovod)

Let's demonstrate a single training step:


In [ ]:
# Single training step demonstration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = custom_model.to(device)
model.train()

# Create dummy batch
batch_size = 4
seq_len = 128
dummy_input_ids = torch.randint(0, config.model.vocab_size, (batch_size, seq_len))
dummy_attention_mask = torch.ones(batch_size, seq_len)
dummy_labels = dummy_input_ids.clone()

# Move to device
input_ids = dummy_input_ids.to(device)
attention_mask = dummy_attention_mask.to(device)
labels = dummy_labels.to(device)

# Forward pass
print("=" * 60)
print("Single Training Step Demonstration")
print("=" * 60)

logits, loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
print(f"[OK] Forward pass complete")
print(f"  Input shape: {input_ids.shape}")
print(f"  Logits shape: {logits.shape}")
print(f"  Loss: {loss.item():.4f}")
print(f"  Perplexity: {compute_perplexity(loss.item()):.2f}")

# Backward pass
loss.backward()
print(f"[OK] Gradients computed")

# Gradient clipping
max_grad_norm = config.training.max_grad_norm
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
print(f"[OK] Gradient clipping applied (norm: {grad_norm:.4f})")

# Optimizer step (simulated)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.training.learning_rate)
optimizer.step()
optimizer.zero_grad()
print(f"[OK] Parameters updated")
print(f"[OK] Training step complete!")


### 5.2 Learning Rate Schedule

The training uses a linear warmup + linear decay schedule:

```python
def get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            # Linear warmup
            return float(current_step) / float(max(1, num_warmup_steps))
        # Linear decay
        return max(0.0, float(num_training_steps - current_step) / 
                   float(max(1, num_training_steps - num_warmup_steps)))
    return LambdaLR(optimizer, lr_lambda)
```

Let's visualize the learning rate schedule:


In [ ]:
# Visualize learning rate schedule
from torch.optim.lr_scheduler import LambdaLR

def get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        return max(0.0, float(num_training_steps - current_step) / 
                   float(max(1, num_training_steps - num_warmup_steps)))
    return LambdaLR(optimizer, lr_lambda)

# Simulate scheduler
base_lr = config.training.learning_rate
num_warmup_steps = config.training.warmup_steps
num_training_steps = 10000  # Example total steps

dummy_optimizer = torch.optim.AdamW([torch.tensor(1.0, requires_grad=True)], lr=base_lr)
scheduler = get_linear_schedule_with_warmup(dummy_optimizer, num_warmup_steps, num_training_steps)

# Compute LR for each step
steps = np.arange(num_training_steps)
learning_rates = []
for step in steps:
    scheduler.step()
    learning_rates.append(scheduler.get_last_lr()[0])

# Plot
plt.figure(figsize=(12, 5))
plt.plot(steps, learning_rates, linewidth=2)
plt.axvline(x=num_warmup_steps, color='r', linestyle='--', label=f'Warmup end ({num_warmup_steps} steps)')
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('Learning Rate', fontsize=12)
plt.title('Learning Rate Schedule (Linear Warmup + Linear Decay)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Initial LR: {base_lr}")
print(f"Warmup steps: {num_warmup_steps}")
print(f"Max LR (at warmup end): {learning_rates[num_warmup_steps]:.6f}")
print(f"Final LR: {learning_rates[-1]:.6f}")


## 6. Horovod Distributed Training

The key to distributed training is **Horovod**, which synchronizes gradients across multiple GPUs.

### 6.1 Horovod Setup

```python
import horovod.torch as hvd

# Initialize Horovod
hvd.init()

# Get rank and world size
rank = hvd.rank()          # Process rank (0, 1, 2, ...)
world_size = hvd.size()    # Total number of processes
local_rank = hvd.local_rank()  # GPU index on this node

# Set CUDA device
torch.cuda.set_device(local_rank)
device = torch.device(f'cuda:{local_rank}')
```

### 6.2 Distributed Optimizer

```python
# Wrap optimizer with Horovod DistributedOptimizer
optimizer = hvd.DistributedOptimizer(
    optimizer,
    named_parameters=model.named_parameters()
)

# Broadcast initial parameters from rank 0
hvd.broadcast_parameters(model.state_dict(), root_rank=0)
hvd.broadcast_optimizer_state(optimizer, root_rank=0)
```

### 6.3 Distributed Data Loading

```python
from torch.utils.data import DistributedSampler

# Create DistributedSampler
sampler = DistributedSampler(
    dataset,
    num_replicas=world_size,
    rank=rank,
    shuffle=True
)

# Create DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=per_gpu_batch_size,
    sampler=sampler,
    shuffle=False  # Shuffling handled by sampler
)
```

### 6.4 Metric Averaging

```python
# Average metrics across all processes
def metric_average(value, name='metric'):
    tensor = torch.tensor(value)
    avg_tensor = hvd.allreduce(tensor, name=name, average=True)
    return avg_tensor.item()

# Usage
loss_value = loss.item()
avg_loss = metric_average(loss_value, name='train_loss')
```

**Key Benefits:**
- **Data Parallelism**: Each GPU processes different batches
- **Gradient Synchronization**: Gradients averaged across GPUs
- **Scalability**: Linear speedup with number of GPUs
- **Fault Tolerance**: Can handle node failures gracefully


## 7. Training Execution

The actual training is executed via command line:

### Local Training (2+ GPUs)
```bash
horovodrun -np 4 -H localhost:4 \
    python -m src.train --config configs/base.yaml
```

### Cluster Training (Zaratan HPC)
```bash
sbatch scripts/train_zaratan.sh configs/base.yaml
```

### Training Output

During training, you'll see:
- **Loss and perplexity** logged every `log_interval` steps
- **Validation** performed every epoch
- **Checkpoints** saved periodically and for best model
- **TensorBoard logs** for visualization

Example output:
```
Epoch 1 | Step 100/5000 | Loss: 8.2341 | PPL: 3782.45 | LR: 0.000500 | Steps/s: 2.34
Epoch 1 | Step 200/5000 | Loss: 7.8912 | PPL: 2654.23 | LR: 0.001000 | Steps/s: 2.35
...
Validation (Epoch 1):
  - Val Loss: 7.5432
  - Val Perplexity: 1890.12
  - Val Accuracy: 0.2345
New best model saved (val_loss: 7.5432)
```


## 8. Results Visualization

After training, you can visualize results using TensorBoard:

```bash
tensorboard --logdir runs/
```

Or load and plot training logs programmatically:


In [ ]:
# Example: Plot training curves (if logs exist)
import glob
import re

def parse_training_logs(log_dir="logs"):
    """Parse training logs to extract metrics."""
    log_files = glob.glob(f"{log_dir}/train_*.log")
    
    if not log_files:
        print("No training logs found. Run training first!")
        return None
    
    # This is a simplified parser - actual implementation would be more robust
    print(f"Found {len(log_files)} log file(s)")
    print("To visualize training curves:")
    print("  1. Run training: sbatch scripts/train_zaratan.sh")
    print("  2. Use TensorBoard: tensorboard --logdir runs/")
    print("  3. Or parse logs programmatically")
    
    return None

# Check for logs
log_dir = project_root / "logs"
if log_dir.exists():
    parse_training_logs(str(log_dir))
else:
    print("Log directory not found. Training logs will be created during training.")
    print("\nExpected log structure:")
    print("  logs/train_YYYYMMDD_HHMMSS.log")
    print("  runs/experiment_name/  (TensorBoard logs)")


## 9. Text Generation

After training, we can generate text from the model. Let's demonstrate the generation process:


In [ ]:
# Text generation demonstration
print("=" * 60)
print("Text Generation Demonstration")
print("=" * 60)

# Load tokenizer
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# Example prompt
prompt = "Once upon a time"
print(f"\nPrompt: '{prompt}'")

# Tokenize prompt
inputs = tokenizer(prompt, return_tensors='pt')
input_ids = inputs['input_ids'].to(device)

print(f"Tokenized: {input_ids[0].tolist()}")
print(f"Decoded: {tokenizer.decode(input_ids[0])}")

# Generate with model (using untrained model for demo)
model.eval()
with torch.no_grad():
    # Generate tokens
    generated_ids = model.generate(
        input_ids=input_ids,
        max_new_tokens=50,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id
    )
    
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    print(f"\nGenerated text (untrained model - will be random):")
    print(f"  {generated_text}")

print("\n" + "=" * 60)
print("Note: This is an untrained model demonstration.")
print("After training, use:")
print("  python -m src.generate --checkpoint checkpoints/best_model.pt \\")
print("                          --config configs/base.yaml \\")
print("                          --prompt 'Your prompt here' \\")
print("                          --interactive")
print("=" * 60)


### 9.1 Generation Strategies

The model supports multiple sampling strategies:

1. **Temperature Sampling**: Controls randomness
   - `temperature < 1.0`: More deterministic, focused
   - `temperature > 1.0`: More random, diverse

2. **Top-k Sampling**: Keep only top k most likely tokens
   - Reduces low-probability tokens
   - `top_k=50`: Consider only top 50 tokens

3. **Top-p (Nucleus) Sampling**: Keep tokens with cumulative probability ≥ p
   - Dynamic vocabulary size
   - `top_p=0.9`: Keep tokens until cumulative prob ≥ 0.9

Example usage:
```python
generated = model.generate(
    input_ids=prompt_ids,
    max_new_tokens=100,
    temperature=0.8,    # Moderate randomness
    top_k=50,          # Top 50 tokens
    top_p=0.9,         # Nucleus sampling
    eos_token_id=tokenizer.eos_token_id
)
```


## 10. Checkpoint Management

The training script saves checkpoints at multiple points:

1. **Periodic Checkpoints**: Every `save_interval` steps
2. **Best Model**: When validation loss improves
3. **Epoch Checkpoints**: At the end of each epoch
4. **Final Model**: At the end of training

Checkpoint structure:
```python
checkpoint = {
    'epoch': epoch,
    'step': step,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),  # Added in our implementation
    'loss': loss,
}
```

### Loading a Checkpoint

```python
checkpoint = torch.load('checkpoints/best_model.pt', map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch'] + 1
```

### Resuming Training

```bash
python -m src.train --config configs/base.yaml \
                    --resume checkpoints/checkpoint_epoch_2.pt
```


## 11. Key Implementation Details

### 11.1 Adaptive GPU Logic

The training script automatically adapts to available resources:

```python
if world_size < 2:
    # Run smoke test instead of full training
    run_smoke_test()
    sys.exit(0)
else:
    # Full distributed training
    run_distributed_training(config, rank, world_size, local_rank)
```

This ensures the code never fails due to insufficient GPUs.

### 11.2 Device Management

Each process uses its assigned GPU:

```python
# Set device based on local_rank
device = torch.device(f'cuda:{local_rank}')
torch.cuda.set_device(local_rank)

# Move model and data to correct device
model = model.to(device)
input_ids = batch['input_ids'].to(device)
```

### 11.3 Rank-Aware Operations

Only rank 0 performs certain operations:

```python
if rank == 0:
    # Save checkpoint
    save_checkpoint(...)
    
    # Write to TensorBoard
    tb_logger.add_scalar(...)
    
    # Write to log file
    logger.info(...)
```

### 11.4 Distributed Sampler Epoch Setting

For proper shuffling across epochs:

```python
train_sampler.set_epoch(epoch)
```

This ensures each epoch sees different data partitions.


## 12. Summary and Next Steps

### What We've Covered

1. **Configuration Management** - YAML-based config system
2. **Model Architecture** - Custom TransformerLM and HF models
3. **Data Pipeline** - Dataset loading and tokenization
4. **Training Loop** - Forward/backward pass, optimization
5. **Horovod Integration** - Distributed training setup
6. **Learning Rate Scheduling** - Warmup + decay schedule
7. **Text Generation** - Sampling strategies
8. **Checkpoint Management** - Save/load and resume

### Running Full Training

**On Zaratan HPC:**
```bash
# Submit job
sbatch scripts/train_zaratan.sh configs/base.yaml

# Monitor
squeue -u $USER
tail -f logs/horovod_transformer-<JOB_ID>.out

# View TensorBoard (after training)
tensorboard --logdir runs/
```

**Local (if you have 2+ GPUs):**
```bash
horovodrun -np 2 -H localhost:2 \
    python -m src.train --config configs/base.yaml
```

### Expected Results

- **Custom Model (15M params)**: Perplexity ~15-25 after 3 epochs
- **DistilGPT-2 (82M params)**: Perplexity ~10-15 after fine-tuning
- **GPT-2 Small (117M params)**: Perplexity ~8-12 after fine-tuning

*Results depend on dataset size, training time, and hyperparameters.*

### Project Files

- **Training Script**: `src/train.py` - Main distributed training
- **Data Loading**: `src/data.py` - Dataset and DataLoader setup
- **Models**: `src/models/` - Custom and HF model implementations
- **Utilities**: `src/utils/` - Config, logging, distributed helpers
- **Scripts**: `scripts/` - Training scripts for local and cluster
- **Configs**: `configs/` - YAML configuration files

---

**For questions or issues, refer to:**
- `README.md` - Comprehensive documentation
- `QUICKSTART.md` - Quick start guide
- `PROJECT_SUMMARY.md` - Implementation summary
